# 01 · Checkpoint CP1: data integrity and leakage (Milestone 2)

Runs every CP1 check of spec section 9.1 (revision 49) on the real data, with
`eviground.data.cp1.run_cp1()`. CPU runtime; no GPU needed.

**What it does**
1. `ensure_data('cp1')`: the four verified datasets are skipped (their `_DONE` is valid); **COCO
   val2017** is downloaded once to `datasets/06_COCO/` (`val2017.zip` 0.8 GB + annotations
   0.25 GB, resumable, CRC-tested). Then everything is unpacked or copied to the local disk
   (SD v1.4 tars ≈ 23 GB, DiffSeg30k zips ≈ 26 GB, SID-Set ≈ 21 GB, COCO 0.8 GB; GenImage
   Validation is read in place on Drive). Free space is checked first.
2. `run_cp1()`: scans every image once (format, JPEG quality, subsampling, size, file /
   pixel / perceptual hashes, DiffSeg30k mask values), reserves the 150 Claims-Edit COCO photos,
   drops COCO negatives that nearly duplicate a test image, makes the 10% stratified validation
   split (seed 42), checks exact and near duplicates, compares class counts with the manifests,
   measures the GenImage real JPEG quality distribution, plans the harmonization
   (spec section 8 table), draws 3,000 crops per class with the training sampler, and runs the
   metadata classifier and the blockiness check. The scan is saved in parts under
   `Paper 1/cache/cp1/`: after a disconnect, re-run Cells 1–4 and it continues.

**Result** (send this file to Claude): `Paper 1/runs/checkpoints/cp1_data_integrity.json`, plus
`runs/checkpoints/cp1/` with the 4×4 sample grids per class and the full manifest
(`manifest_cp1.csv.gz`). The verdict printed at the end is only the script's; the checkpoint is
recorded in `docs/CHECKPOINTS.md` from the JSON.

**Secrets**: `GITHUB_TOKEN` only (clone while private). COCO needs no account.


In [ ]:
# Cell 1: mount Drive, get the code, install it
import os, subprocess, sys

DRY = os.environ.get('EVIGROUND_DRY_RUN') == '1'   # CI / local test: fake sources, temp folders
REPO_URL = 'github.com/DHANUSHKUMMARI/Paper1.git'
BRANCH = 'main'                                    # 'claude/nice-curie-3tzxzr' before the merge

if DRY:
    REPO = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
else:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    REPO = '/content/Paper1'
    if not os.path.exists(REPO):
        try:
            tok = userdata.get('GITHUB_TOKEN')       # never printed
        except Exception:
            tok = None
        url = f'https://{tok}@{REPO_URL}' if tok else f'https://{REPO_URL}'
        r = subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, REPO],
                           capture_output=True, text=True)
        if r.returncode != 0:                        # stderr is not shown: it can contain the URL
            raise RuntimeError('git clone failed: check GITHUB_TOKEN (Secrets panel) and BRANCH')
        subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'],
                       check=True)
    else:
        subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO], check=True)

sys.path.insert(0, os.path.join(REPO, 'src'))
# Re-running in a runtime that already imported eviground would keep the OLD code in memory
# although git pulled the new one: drop it so the next import reads the files on disk.
for _name in [m for m in sys.modules if m == 'eviground' or m.startswith('eviground.')]:
    del sys.modules[_name]
print('repo:', REPO, '| branch:', BRANCH, '| dry run:', DRY)

In [ ]:
# Cell 2: settings; free space before anything starts
import json, logging, os, time
logging.basicConfig(level=logging.INFO, force=True)
from eviground.data.storage import free_bytes
from eviground.env import drive_quota, get_paths, session_info

CU_PER_HOUR = None      # optional: the CPU runtime's rate from Colab's resource panel
RESERVE_GB = 5.0        # always left free on Drive and on the local disk
WORKERS = os.cpu_count() or 2
CROPS_PER_CLASS = 3000  # sampler draws per class for the metadata classifier

T0 = time.time()
info = session_info(cu_per_hour=CU_PER_HOUR)
paths = get_paths()
if DRY:   # a fresh fake Drive of its own (other notebooks' dry runs use the default temp folder)
    import shutil, tempfile
    from pathlib import Path
    from eviground.env import Paths
    _dry = Path(tempfile.gettempdir()) / 'eviground_cp1_dry'
    shutil.rmtree(_dry, ignore_errors=True)
    paths = Paths(root=_dry / 'drive' / 'Paper 1', local_data=_dry / 'data')
paths.ensure()
def disk_now():
    q = drive_quota()   # the Drive mount reports this machine's disk, so read the quota instead
    return {'drive_quota_free_gb': None if not q or q['free'] is None else round(q['free'] / 1e9, 1),
            'local_free_gb': round(free_bytes(paths.local_data) / 1e9, 1)}
disk_before = disk_now()
import eviground.data.cp1 as _cp1
print('code:', _cp1.__file__, '| commit', info['git_commit'], '| workers', WORKERS)
print('free space:', disk_before)


In [ ]:
# Cell 3: verify the datasets, download COCO once, stage everything on the local disk
from eviground.data.download import ensure_data

if DRY:   # a small fake Drive with the layouts the probe found; COCO from a fake URL source
    from eviground.data.fake_cp1 import cp1_overrides, make_cp1_world
    DRY_WORLD = dict(sd14_per_class=30, val_per_class=2, diffseg=(36, 12),
                     sid_caps={'train': 24, 'validation': 6}, coco_images=60)
    sources = make_cp1_world(paths.datasets, paths.root.parent / 'fake_remote', **DRY_WORLD)
    overrides = cp1_overrides(DRY_WORLD['coco_images'])
else:
    sources, overrides = None, None

report = ensure_data('cp1', paths=paths, sources=sources, reserve_gb=RESERVE_GB,
                     stage_local=True, raise_errors=False, overrides=overrides)
for key, entry in report.items():
    staged = entry.get('staged') or {}
    print(f"{key:<14} {entry['status']:<9} staged: {staged.get('files', staged.get('reason', ''))} "
          f"{entry.get('error', '')}")
failed = [k for k, v in report.items() if v['status'] == 'error']
assert not failed, f'not ready: {failed} (see the errors above; re-run this cell to resume)'


In [ ]:
# Cell 4: run every CP1 check (resumes the image scan from Paper 1/cache/cp1)
from eviground.data.cp1 import CP1Config, run_cp1

cfg = CP1Config(workers=WORKERS, crops_per_class=CROPS_PER_CLASS)
if DRY:
    cfg = CP1Config(workers=2, crops_per_class=150, blockiness_per_group=40,
                    diffseg_card_splits={'train': DRY_WORLD['diffseg'][0],
                                         'validation': DRY_WORLD['diffseg'][1]})
OUT = paths.runs / 'checkpoints' / 'cp1'
result = run_cp1(paths, cfg, out_dir=OUT, cache_dir=paths.cache / 'cp1')
print('verdict (script):', result['verdict'])
print('criteria:', result['criteria'])
print('stop and ask:', result['stop_and_ask'])


In [ ]:
# Cell 5: save the record to Drive and print the numbers that decide CP1
import datetime

hours = (time.time() - T0) / 3600
record = {
    'checkpoint': 'CP1',
    'utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
    'dry_run': DRY,
    'branch': BRANCH,
    'session': info,
    'disk_before': disk_before,
    'disk_after': disk_now(),
    'wall_time_h': round(hours, 3),
    'cu_per_hour': CU_PER_HOUR,
    'cu_estimate': round(hours * CU_PER_HOUR, 3) if CU_PER_HOUR else None,
    'ensure_data': {k: {kk: v.get(kk) for kk in ('status', 'error', 'staged')} for k, v in report.items()},
    'result': result,
}
path = paths.runs / 'checkpoints' / 'cp1_data_integrity.json'
path.write_text(json.dumps(record, indent=1, default=str))
print('saved', path, '| wall time', round(hours * 60, 1), 'min')

m = result['metadata_classifier']
print('\nmetadata classifier (format, JPEG quality, crop size), balanced accuracy:',
      m['criterion_score'], '(<= 0.60 required)')
print('  single features:', m['criterion']['real_vs_not_real']['single_feature'])
print('  diagnostics:', m['diagnostics'])
print('exact duplicates across splits:', result['exact_duplicates']['cross_split_total'])
for role, v in result['near_duplicates'].items():
    if isinstance(v, dict) and v.get('images'):
        print(f"near-duplicates {role}: {v['counts']} of {v['images']} (share at <=6: {v['share_primary']})")
print('class counts equal the manifests:', result['class_counts']['all_equal'])
for t in result['class_counts']['table']:
    if not t['equal']:
        print('  differs:', t)
print('blockiness (harmonized, median per source/class):',
      {k: v.get('median') for k, v in result['blockiness']['harmonized'].items()})
print('blockiness separation:', result['blockiness']['separation'])
print('DiffSeg30k mask values:', result['diffseg30k_masks']['distinct_values'])
print('COCO negatives dropped as near-duplicates of a test image:',
      result['coco_negatives']['near_dup_filter']['dropped'])
